# KyVector to AGOL Overwrites

This notebook is a template for the overwrite to AGOL process with **multiple** feature classes.

If this project errors with something to the affect of *cannot create folder, Access Denied*, the safest course of action is to shutdown pro, and then kill any orphaned processes in the task manager.  That should clear most of the issues.  

As a best practice, credentials are stored in a *constants.py* that will be imported with the modules.

In [1]:
# import modules
import os
import arcpy
import shutil
import pandas as pd
from pathlib import Path
from zipfile import ZipFile
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection

# replace creds_example with creds
from creds import (
    CLIENT_ID,
    DB_USER,
    DB_PASSWORD,
    DB_CONNECTION_NAME,
    INSTANCE,
    DATABASE,
    DATABASE_PLATFORM
)

arcpy.env.overwriteOutput = True   
arcpy.env.addOutputsToMap = False  # prevents File Geodatabase locks.


### Global Variables

In [2]:
# AGOL login. Will launch a browser window for OAuth2 authentication.
# Select the appropriate user account when prompted.
gis = GIS(url="https://wwww.arcgis.com", client_id=CLIENT_ID)

 
db_user = DB_USER
db_password = DB_PASSWORD
instance = INSTANCE
database = DATABASE
database_platform = DATABASE_PLATFORM
OUT_FOLDER_PATH = "."

# projection variables
out_crs = arcpy.SpatialReference('WGS_1984_Web_Mercator_Auxiliary_Sphere')
transformation_values = 'WGS_1984_(ITRF00)_To_NAD_1983'

gis


Please sign in to your GIS and paste the code that is obtained below.
If a web browser does not automatically open, please navigate to the URL below yourself instead.
Opening web browser to navigate to: https://www.arcgis.com/sharing/rest/oauth2/authorize?response_type=code&client_id=W07x5VpXOQryqIOC&redirect_uri=urn%3Aietf%3Awg%3Aoauth%3A2.0%3Aoob&state=y3rHVhpMevkUVyTJDP61LlwmdcrfOm&allow_verification=false


GIS @ https://wwww.arcgis.com version:[2026, 1]

### Map the feature classes to services

In [3]:
db_connection_name = DB_CONNECTION_NAME
DB_CONNECTION = Path(f'{db_connection_name}.sde')

if DB_CONNECTION.exists():
    print(f'"{DB_CONNECTION}" already exist.') 

else:
    arcpy.management.CreateDatabaseConnection(
        out_folder_path=OUT_FOLDER_PATH,
        out_name=DB_CONNECTION_NAME,
        database_platform="SQL_SERVER",
        instance=instance,
        account_authentication="DATABASE_AUTH",
        username=db_user,
        password=db_password,
        save_user_pass="SAVE_USERNAME",
        database=database
    )

    print(f'Connection file {DB_CONNECTION} created.')


Connection file kyvector_kysdestaging_kyvector.sde created.


### Set workspace 

This needs to go the the feature dataset of interest.

In [ ]:
arcpy.env.workspace = f'{DB_CONNECTION}/kyvector.SDE.institutional'  # change this to your schema and       
                                                                     # feature dataset if applicable
feature_classes = arcpy.ListFeatureClasses()
feature_classes

['kyvector.SDE.Prisons',
 'kyvector.SDE.Structure_Points',
 'kyvector.SDE.KYNATIONALGUARD_ARMORIES_BOUNDARIES',
 'kyvector.SDE.Libraries',
 'kyvector.SDE.KYNATIONALGUARD_ARMORIES',
 'kyvector.SDE.KY_Opportunity_Zones',
 'kyvector.SDE.Hospitals',
 'kyvector.SDE.Industrial_Site_Boundaries',
 'kyvector.SDE.Schools',
 'kyvector.SDE.Industrial_Site_Points',
 'kyvector.SDE.Postsecondary_Schools',
 'kyvector.SDE.City_Halls',
 'kyvector.SDE.Industrial_Site_Tracts',
 'kyvector.SDE.Public_School_Buffers',
 'kyvector.SDE.County_Courthouses',
 'kyvector.SDE.Available_Industrial_Buildings',
 'kyvector.SDE.Manufacturing_Establishments',
 'kyvector.SDE.Long_Term_Care',
 'kyvector.SDE.Public_School_Driving_Distances']

In [5]:
df = pd.DataFrame(feature_classes, columns=['feature_class'])
df['item_id'] = ''
df['item_title'] = ''
df['fgdb_name'] = ''
df.columns


Index(['feature_class', 'item_id', 'item_title', 'fgdb_name'], dtype='str')

#### create a subset to process

The mapping will link item id, title and gdb basename to the feature class.  It may be more efficient to import the mapping like we did the creds.

In [6]:
mapping = {
  'kyvector.SDE.Schools': ('d1c8d1da34af465c8165e8d4cc661f0d', 'Kentucky Public Schools','Ky_Public_Schools_WM'),
  'kyvector.SDE.Public_School_Buffers':   ('06558648bb924b73b8d195a7774296b9','Kentucky Public Schools with 0.5 mi and 1 mi buffers','Ky_Public_School_Buffers_WM'),
  'kyvector.SDE.Public_School_Driving_Distances': ('f39248f4bdeb41199ff6e9fe6375a8fb', 'Kentucky Public School Driving Distances', 'Ky_Public_School_Driving_Distances_WM')
}
for fc, (iid, name, gdb) in mapping.items():
    df.loc[df['feature_class'] == fc, ['item_id','item_title','fgdb_name']] = (iid, name, gdb)

vals = [
    'kyvector.SDE.Schools',
    'kyvector.SDE.Public_School_Buffers',
    'kyvector.SDE.Public_School_Driving_Distances'
]
df_subset = df[df['feature_class'].isin(vals)]
df_subset


,feature_class,item_id,item_title,fgdb_name
8,kyvector.SDE.Schools,d1c8d1da34af465c8165e8d4cc661f0d,Kentucky Public Schools,Ky_Public_Schools_WM
13,kyvector.SDE.Public_School_Buffers,06558648bb924b73b8d195a7774296b9,Kentucky Public Schools with 0.5 mi and 1 mi b...,Ky_Public_School_Buffers_WM
18,kyvector.SDE.Public_School_Driving_Distances,f39248f4bdeb41199ff6e9fe6375a8fb,Kentucky Public School Driving Distances,Ky_Public_School_Driving_Distances_WM


#### Save to file (optional)

If you want to save to file so you can later update the script to pull from file instead of hard coding values, run this next cell.

In [7]:
os.makedirs('input_tables', exist_ok=True)
df_subset.to_csv('input_tables/kyvector_schools.csv', index=False)
df_subset.to_json('input_tables/kyvector_schools.json', orient='records')
# df_subset.to_excel('input_tables/kyvector_schools.xlsx', index=False)
# df_subset.to_parquet('input_tables/kyvector_schools.parquet', index=False)

## Functions

Set up Project Feature Class. Basic maintenance:

 - clean/delete paths
 - create new path names
 - create file geodatabase
 - project feature class into file geodatabase

In [8]:
# Create functions for various steps
# step 1 - Project to web mercator
def project_fc(row):
    """
    This function will project the input feature class into
    web mercator.

    1. Check to see if gdb and zip files exist - delete
    2. Create new File Geodabatase
    2. Project
    3. compress to zip file
    """

    # deleting existing FGDB if necessary
    if os.path.exists(gdb_location):
        shutil.rmtree(gdb_location)
    if os.path.exists(zipped_file):
        os.remove(zipped_file)

    # create File Geodatabse
    # arcpy.AddMessage(f'Creating {output_gdb}.gdb')
    print(f'Creating "{output_gdb}.gdb"')

    arcpy.CreateFileGDB_management(out_folder_path=output_folder,
                                   out_name=output_gdb)

    # arcpy.AddMessage(f'Successfully created {output_gdb}.gdb')
    print(f'Successfully created "{output_gdb}.gdb"')

    # project to web mercator
    arcpy.Project_management(in_dataset=input_data,
                             out_dataset=out_fc_path,
                             out_coor_system=out_crs)

    # arcpy.AddMessage(f'Successfully completed Project process: {out_fc} created.\n'
    #                  f'Compact, clear cache, and zip {output_gdb}.gdb')
    print(f'Successfully completed Project process: "{out_fc}" created.\n'
          f'Compact, clear cache, and zip "{output_gdb}.gdb"')

    # Compact and Clear Cache - removes locks
    arcpy.Compact_management(gdb_location)
    arcpy.ClearWorkspaceCache_management(gdb_location)

    # Zip file geodatabase
    with ZipFile(zipped_file, 'w') as myzip:
        for root, dirs, files in os.walk(gdb_location):
            root = str(root)
            for file in files:
                # skip ArcGIS lock files
                if file.endswith(".lock"):
                    continue

                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, os.path.dirname(gdb_location))
                myzip.write(full_path, arcname=rel_path)

    # arcpy.AddMessage(f'{output_gdb}.gdb successfully zipped')
    print(f'"{output_gdb}".gdb successfully zipped')

    # arcpy.AddMessage('Project process complete'
    #                  'Time to overwrite agol item\n')
    print(f'Project process complete\n'
          f'\nTime to overwrite agol item\n')

    return gdb_location, zipped_file


### Overwrite Function

This function will take output from the `project_fc()` function (gdb_location, zipped_file) as sources to use to overwrite the AGOL Item.

 - get AGOL item
 - save thumbnail
 - convert item to FeatureLayerCollection
 - overwrite with zipfile
 - get AGOL item 2nd time
 - update the item title and item thumbnail

In [9]:

# Create a function to overwrite item on agol
def overwrite_item():
    """
    This function takes the zip file created in the previous function,
    turns it into a FeatureLayerCollection, and then overwrites the
    item on AGOL.

    During the process, the function grabs the item thumbnail prior to
    the overwrite step and saves it in memory.  Once item is overwritten,
    the script updates the item with the in memory png file.
    """

    # call project fc()
    gdb_location, zipped_file = project_fc(row)
    print("Created:", gdb_location, zipped_file)
    
    # get some info first
    item = gis.content.get(agol_item_id)
    # arcpy.AddMessage(f'Getting ready to overwrite {item.title} with {zip_file_name}\n')
    print(f'Getting ready to overwrite "{item.title}" with "{zip_file_name}"')

    # use the item id to get the existing item
    # arcpy.AddMessage(f'Set item and download thumbnail.')
    print(f'Set item and download thumbnail.')
    existing_item = item

    # save the thumbnail in memory
    item_thumbnail = existing_item.download_thumbnail('in_memory')
    # arcpy.AddMessage(f'Item thumbnail downloaded.\n'
    #                  f'Create a FeatureLayerCollection of {agol_item_title} and overwrite the item from {zip_file_name}.')
    print(f'Item thumbnail downloaded.\n'
          f'Create a FeatureLayerCollection of "{agol_item_title}"\n'
          f'and overwrite the item from "{zip_file_name}".')

    # create a # Create a Feature Layer Collection from Item and overwrite with zipped geodatabase
    flayer_collection = FeatureLayerCollection.fromitem(existing_item)
    flayer_collection.manager.overwrite(zipped_file)
    # arcpy.AddMessage(f'Item {agol_item_title} successfully overwritten.\n'
    #                  f'Updating item title and thumbnail')
    print(f'Item "{agol_item_title}" successfully overwritten.\n'
          f'Updating item title and thumbnail')

    # update title and thumbnail
    # need to make another get requests to get updated info
    updated_item = gis.content.get(agol_item_id)

    updated_item.update(item_properties={'title': agol_item_title})
    # arcpy.AddMessage(f'Successfully updated item title.')
    print(f'Successfully updated item title.')

    updated_item.update(thumbnail=item_thumbnail)
    # arcpy.AddMessage(f'Successfully updated item thumbnail.\n'
    #                  f'Overwrite process complete\n')
    print(f'Successfully updated item thumbnail.\n'
          f'Overwrite process complete\n\n\n'
          f'##############################################################################\n\n\n'                                  
)

    return updated_item


In [10]:
# agol_item_id    output_gdb  input_data  fc_path  zipped_file  out_fc
# zip_file_name agol_item_title 
        
output_folder = 'FGDBs'
os.makedirs(output_folder, exist_ok=True)

for _, row in df_subset.iterrows():

    """ 
    Parameter valuables copied from standalone script
    """
    # # User input variables
    # input_data = arcpy.GetParameterAsText(0)
    # output_folder = arcpy.GetParameterAsText(1)
    # output_gdb = arcpy.GetParameterAsText(2)
    # agol_item_id = arcpy.GetParameterAsText(3)
    # agol_item_title = arcpy.GetParameterAsText(4)

    # standalone variable converted to coded variables to processing
    # Index(['feature_class', 'item_id', 'item_title', 'fgdb_name'], dtype='str')

    try:
        input_data = row['feature_class']
        output_gdb =  row['fgdb_name']
        agol_item_id = row['item_id']
        agol_item_title = row['item_title']
            
        # variables derived from input variables
        gdb_location = os.path.join(output_folder, f'{output_gdb}.gdb')
        out_fc = output_gdb  # feature class name
        out_fc_path = os.path.join(gdb_location, out_fc)
        zipped_file = f'{gdb_location}.zip'
        zip_file_name = os.path.basename(zipped_file)

        #  Overwrite the AGOL item
        updated_item = overwrite_item()

    except Exception as e:
        print(f"ERROR processing {row.get('input_data')}: {e}")
    


Creating "Ky_Public_Schools_WM.gdb"
Successfully created "Ky_Public_Schools_WM.gdb"
Successfully completed Project process: "Ky_Public_Schools_WM" created.
Compact, clear cache, and zip "Ky_Public_Schools_WM.gdb"
"Ky_Public_Schools_WM".gdb successfully zipped
Project process complete

Time to overwrite agol item

Created: FGDBs\Ky_Public_Schools_WM.gdb FGDBs\Ky_Public_Schools_WM.gdb.zip
Getting ready to overwrite "Kentucky Public Schools" with "Ky_Public_Schools_WM.gdb.zip"
Set item and download thumbnail.
Item thumbnail downloaded.
Create a FeatureLayerCollection of "Kentucky Public Schools"
and overwrite the item from "Ky_Public_Schools_WM.gdb.zip".
Item "Kentucky Public Schools" successfully overwritten.
Updating item title and thumbnail
Successfully updated item title.
Successfully updated item thumbnail.
Overwrite process complete


##############################################################################



Creating "Ky_Public_School_Buffers_WM.gdb"
Successfully created "Ky_